## DSAN 6000 Homework 4A: I/O Speed and Memory Efficiency with DuckDB

## Overview

In the very first week of class, when you first saw the following diagram:

<center>

<img src='images/big-data-definition.svg' width='60%'></img>

</center>

You may have had your Pandas hat on, and you may have thought (as I did when I first learned this material!) something like:

> *Wait... if I want to download and analyze a data file from some website, but I hit the wall that the data file is too big to be **stored** on a single computer... am I not just out of luck?*
> 
> *Because, in the abstract, I'd love to "chop" this data file into smaller pieces. But in practice, wouldn't I need to **download it** onto a computer in the first place, so that this computer could then run some **code** or **process** to actually **do** the "chopping"?*

After some deeper-diving and/or asking AI things about that question, you would hopefully resolve these fears a bit, by learning how:

* (a) Functions like `pd.read_csv()` *do* have arguments like `chunksize` that could allow reading a file in pieces, and
* (b) These functions also support reading from **remote** sources like URLs or S3 bucket URIs.

Nevertheless, you would still be faced with the somewhat-daunting problem of figuring out what value to use for the `chunksize` argument, since different data values generally require different amounts of storage space: if you were downloading a `.csv` file where each row contained the text of a historical novel, for example, your choice of `chunksize` might force Pandas to stop reading right in the middle of the single row containing the 1.2 million words of Proust's [*À la Recherche du temps perdu*](https://observablehq.com/@guillaume-lesaine/in-search-of-marcel-proust) 😱

The point of this introduction is just to scare you away from "manually" optimizing chunk-by-chunk reading of `.csv` files, and towards embracing the combination of the `.parquet` format and DuckDB, given one of DuckDB's most powerful features: the ability to

* **Read file metadata** before loading the actual contents of the file, and then
* **Use** this metadata to **optimize the loading and processing of the file's contents.**

With this combination, you can "hand" a query over to DuckDB (note the full separation of computation from data that this implies!), and it will figure out how to optimize RAM usage on your computer, by loading **only the specific *subset* of the full data file content** that is necessary to satisfy the query.

In Part 1 you will see what this looks like concretely, for the ACLED event data mentioned in the `README.md` overview.

## Part 1: Querying File *Metadata* From S3

As mentioned in class during Week 4, one of the biggest drawbacks of the `.csv` format from a *Data Engineering* perspective is its lack of **embedded metadata** that could be used to plan out the optimal execution of a query *in advance*.

Given just a `.csv` file, a Data Engineer has no way of knowing (for example) whether they can immediately start computing means and averages of certain columns as rows are being read into memory line-by-line, or whether some conversion (say, parsing of timestamps) or missing-data handling will have to be carried out first, *after* the entire file has been read into memory.

As was *also* mentioned in Week 4, however, data files in the `.parquet` format always include **built-in** schema information, in the form of **metadata** stored at the end of the file.

> **The Parquet Metadata *Footer***
> 
> If you're curious, there's a technical reason for storing metadata at the *end* rather than the beginning of the `.parquet` file: storage setups like S3 only support **sequential writes** and **don't** support "editing" or "updating" the contents of a file after its bytes have been written (all files in S3 are immutable). Thus, efficient creation of a `.parquet` file from some prior data format involves (1) writing the data values into an S3 object as bytes, sequentially, computing statistics about these bytes along the way, and then (2) writing the final statistics – like the number of missing values – at the very end.
> 
> In other words, given how S3 objects work, there is no way to "go back up" to the top of the file to write this metadata, since data objects stored in S3 in fact have no "edit" or "update" mode at all. If you use `boto3` or some other API to "add" bytes to the beginning of an S3 object, what is really happening under the hood is that an **entirely new file** is being created: the bytes you're hoping to add to the "beginning" of the original file are then written first, followed by the remaining bytes from the original file, and the resulting *new* file is given the same name as the previous file (which is then discarded, unless you're paying extra for S3's version control features!)

Using DuckDB in combination with data stored in the `.parquet` format, therefore, you can quickly **query just the *metadata*** of the file you eventually hope to process. Run the following cell to see an ultra-simple example (where we just write a test file and then immediately query its metadata), then proceed to Question 1 after you understand this example.

In [35]:
#| label: Q1-example
import pandas as pd
import numpy as np

test_df = pd.DataFrame({
  'col1': [0,1,2,3,4,5,6,7,8,9],
  'col3': [
    'jeff','samyu','fangzhou','siru',
    '21 savage','42dugg',
    '22 savage','43dugg',
    '23 savage','44dugg'],
  'col4': [3.14,6.28,np.nan,np.nan,21.0,42.0,22.0,43.0,23.0,44.0],
})
display(test_df)
test_df.to_parquet('test.parquet')

,col1,col3,col4
0,0,jeff,3.14
1,1,samyu,6.28
2,2,fangzhou,NaN
3,3,siru,NaN
4,4,21 savage,21.00
5,5,42dugg,42.00
6,6,22 savage,22.00
7,7,43dugg,43.00
8,8,23 savage,23.00
9,9,44dugg,44.00


In [36]:
import duckdb

test_con = duckdb.connect()
test_meta_df = test_con.execute("""
SELECT * FROM parquet_metadata('test.parquet')
""").df()
with pd.option_context('display.max_columns', None):
  display(test_meta_df)

,file_name,row_group_id,row_group_num_rows,row_group_num_columns,row_group_bytes,column_id,file_offset,num_values,path_in_schema,type,stats_min,stats_max,stats_null_count,stats_distinct_count,stats_min_value,stats_max_value,compression,encodings,index_page_offset,dictionary_page_offset,data_page_offset,total_compressed_size,total_uncompressed_size,key_value_metadata,bloom_filter_offset,bloom_filter_length,min_is_exact,max_is_exact,row_group_compressed_bytes,geo_bbox,geo_types
0,test.parquet,0,10,3,506,0,0,10,col1,INT64,0,9,0,<NA>,0,9,SNAPPY,"PLAIN, RLE, RLE_DICTIONARY",<NA>,4,69,146,174,{},<NA>,<NA>,True,True,440,<NA>,<NA>
1,test.parquet,0,10,3,506,1,0,10,col3,BYTE_ARRAY,NaN,NaN,0,<NA>,21 savage,siru,SNAPPY,"PLAIN, RLE, RLE_DICTIONARY",<NA>,150,249,157,178,{},<NA>,<NA>,True,True,440,<NA>,<NA>
2,test.parquet,0,10,3,506,2,0,10,col4,DOUBLE,3.14,44.0,2,<NA>,3.14,44.0,SNAPPY,"PLAIN, RLE, RLE_DICTIONARY",<NA>,307,367,137,154,{},<NA>,<NA>,True,True,440,<NA>,<NA>


### Question 1: Your Turn!

Now that you have a feel for the *format* in which a Parquet presents its metadata to DuckDB, your job is as follows:

1.  Write a DuckDB query to obtain the **metadata for the ACLED events file stored at the following S3 URI:**

    ```
    s3://dsan6000-data/acled_events.parquet
    ```

    and store this obtained metadata in a Pandas `DataFrame` called `acled_meta_df`.
1.  From that metadata, construct a new column in `acled_meta_df` named `compression_ratio`, which you should compute as the ratio of the total *compressed* size to the total *uncompressed* size for each column in the ACLED dataset.
1.  Sort `acled_meta_df` in **increasing** order by `compression_ratio`, so that the column with the **"best" compression** (the column for which the most memory is "saved" via compression) appears first, and the column with the **"worst" compression** appears last, and save this sorted `DataFrame` as `compression_ratios.csv`.
1.  Finally, **group** the rows in `acled_meta_df` by `type`, and then compute the **mean** compression ratio for each type. Which datatype is the `.parquet` format able to compress the most? (Record your answer in the string variable at the end of the code cell).


In [ ]:
#| label: Q1-response

# Your code here: carry out the steps as described above



# Replace with the name of the type with the "best" mean compression ratio:
# For example, if the Parquet file's BYTE_ARRAY columns achieved the best mean
# compression, replace "" with the string "BYTE_ARRAY"
most_compressed_type = ""

## Part 2: Optimizing File *Content* Queries with DuckDB

In [ ]:
#| label: Q2-response


## Part 3: Optimizing Your Pandas Workflow

In [ ]:
#| label: Q3-response
